# Lab 9 — DATS 6103

**Name:**

This notebook mirrors the lab page. Everything here also runs in the browser at
<https://www.smajhi.com/DATS-6103/labs/regression.html>, where the self-checks and the timers live; work in whichever you
prefer. The discussion half of the lab is not in this file — it happens away
from the keyboard.

**Conditions.** No AI assistants. The official Python, NumPy, Pandas,
Matplotlib and scikit-learn documentation and the course notes may be consulted
freely.

Run the self-check under each answer. It tells you whether your answer is right
without telling you what the answer is.

## Part A—Work it out

**Not submitted, and not written at the keyboard.** Work these in your team, on paper, one problem at a time; each is reported to the class as you finish it. The hint cards are on the lab page: <https://www.smajhi.com/DATS-6103/labs/regression.html>.

### A1. 98% accurate

*Suggested: 5 minutes.*

Of 1,000 card transactions, 20 are fraud. A model flags 30, and 15 of those are
fraud.

a. Fill in the confusion matrix, and compute accuracy, precision and recall for
   the fraud class.
b. A second "model" flags nothing at all. What is its accuracy?
c. Which single number would you put on the slide, and why not accuracy?

### A2. Where to put the threshold

*Suggested: 6 minutes.*

Same fraud model. At a threshold of 0.5 it flags 30 transactions and catches 15
frauds. At 0.2 it flags 79 and catches 19. Every flag costs a \$5 manual review;
every fraud that slips through costs \$500.

a. Compute precision and recall at 0.2.
b. Compute the total cost at each threshold. Which do you choose?
c. A colleague objects that precision fell from 50% to 24%. Answer them.

### A3. Reading the coefficients

*Suggested: 6 minutes.*

A simpler fraud model is a logistic regression on two features: the amount, in
hundreds of dollars, and whether the card was used abroad (1 if so, 0 if not).
Its fitted log odds of fraud are
$$
-3 + 0.7 \times \text{amount} + 1.1 \times \text{abroad}.
$$
Take $e^{0.7} \approx 2$, $e^{1.1} \approx 3$ and $e^{-3} \approx 0.05$.

a. Say in one sentence each what 0.7 and 1.1 mean.
b. Find the probability of fraud for a \$300 purchase at home and for a \$300
   purchase abroad. Which does a threshold of 0.5 flag?
c. A colleague says being abroad adds the same amount to the probability of fraud
   at every dollar amount. Test that by comparing \$0 with \$300. What does stay
   the same?

### A4. The winner's number

*Suggested: 5 minutes.*

A team tries 50 values of `C` by 5-fold cross-validation. The best scores 0.91,
and the report says "expected accuracy: 91%." On a fresh test set the model
scores 0.87.

a. Why is the gap in that direction, and not the other?
b. What should the report have said, and when should the test set have been
   touched?
c. The team also standardized all the data, test rows included, before splitting.
   What does that add to the problem?

## Part B—Build

### B1. Which class is positive?

*Suggested: 8 minutes.*

`bc` is the Wisconsin breast cancer data: 569 tumors, 30 measurements each. The
label is `0` for **malignant** and `1` for benign, which is the opposite of the
order most people assume.

`pipe` is written for you: the scaler puts every column on one scale, and then
logistic regression runs. Fit it on the training data to get `model`. Then report,
on the **test** data:

- `acc`---the accuracy;
- `recall_malignant`---the share of malignant tumors the model calls malignant;
- `missed`---how many malignant tumors it calls benign.

**Already loaded:** `bc`, `X_train`, `X_test`, `y_train` and `y_test`.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103, stratify=y)

In [ ]:
# The scaler is given. HW Regression Q4 shows why it matters.
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
model = ...
y_hat = ...
acc = ...
recall_malignant = ...
missed = ...

print("accuracy", acc, "| malignant recall", recall_malignant, "| missed", missed, "of", (y_test == 0).sum())

In [ ]:
# self-check: run this after your answer above
from sklearn.metrics import recall_score
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_train, y_train)
yh = m.predict(X_test)
want = float(recall_score(y_test, yh, pos_label=0))
benign = float(recall_score(y_test, yh))
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("model", model, lambda: np.array_equal(model.predict(X_test), yh),
          "should be pipe fit on X_train and y_train."),
    _part("y_hat", y_hat, lambda: np.array_equal(y_hat, yh), "should be the model's predictions on X_test."),
    _part("acc", acc, lambda: abs(float(acc) - float((yh == y_test).mean())) < 1e-9,
          "should be the test accuracy."),
    _part("recall_malignant", recall_malignant, lambda: abs(float(recall_malignant) - want) < 1e-9,
          "that is the recall for benign: recall_score's default positive class is 1, and malignant is 0."
          if not _blank(recall_malignant) and abs(float(recall_malignant) - benign) < 1e-6
          else "should be the share of truly malignant test tumors predicted malignant."),
    _part("missed", missed, lambda: int(missed) == int(((y_test == 0) & (yh == 1)).sum()),
          "should count tumors that are malignant (0) but predicted benign (1)."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B2. Move the threshold

*Suggested: 10 minutes.*

`predict` uses a probability of 0.5. A missed cancer and a false alarm do not
cost the same, so the threshold is a decision, not a default.

Using `p_malignant`, the model's probability that each test tumor is malignant,
flag a tumor whenever `p_malignant >= t`. For `t` in `0.5`, `0.2` and `0.1`,
count the missed malignant tumors and the false alarms, and score each threshold
by `cost = 50 * missed + false_alarms`.

- `costs`---a dict from each threshold to its cost;
- `best_t`---the threshold with the lowest cost.

**Already loaded:** `X_train`, `X_test`, `y_train` and `y_test`, the fitted `model`, and `p_malignant`.

In [ ]:
# uses model from B1 above: run B1 first

p_malignant = model.predict_proba(X_test)[:, 0]

In [ ]:
costs = {}
for t in [0.5, 0.2, 0.1]:
    flagged = ...
    missed = ...
    false_alarms = ...
    costs[t] = ...
best_t = ...

print(costs, best_t)

In [ ]:
# self-check: run this after your answer above
want = {}
for t_ in [0.5, 0.2, 0.1]:
    fl = p_malignant >= t_
    want[t_] = 50 * int(((y_test == 0) & ~fl).sum()) + int(((y_test == 1) & fl).sum())
fl = p_malignant >= 0.1                    # the loop's last pass is what is left in flagged and the rest
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("flagged", flagged, lambda: np.array_equal(flagged, fl), "should be True where p_malignant >= t."),
    _part("missed", missed, lambda: int(missed) == int(((y_test == 0) & ~fl).sum()),
          "should count malignant tumors (0) that were not flagged."),
    _part("false_alarms", false_alarms, lambda: int(false_alarms) == int(((y_test == 1) & fl).sum()),
          "should count benign tumors (1) that were flagged."),
    _part("costs", costs, lambda: set(costs) == {0.5, 0.2, 0.1} and all(int(costs[k]) == want[k] for k in want),
          "costs[t] should be 50 * missed + false_alarms, for each t."),
    _part("best_t", best_t, lambda: best_t == min(want, key=want.get),
          "should be the threshold with the lowest cost."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B3. The training score lies

*Suggested: 10 minutes.*

Logistic regression's `C` is the inverse of the penalty, as in
[the reading](https://www.smajhi.com/DATS-6103/regularization/regularization.html). For each `C` in `Cs`,
record the **training** accuracy and the **5-fold cross-validated** accuracy on
the training data, using `folds`. Then refit the best `C` by cross-validation and
score it on the test set.

- `train_acc`, `cv_acc`---lists, one entry per `C`;
- `best_C`---the `C` with the highest cross-validated accuracy;
- `test_acc`---that model's accuracy on the test set.

**Already loaded:** `X_train`, `X_test`, `y_train` and `y_test`, `folds`, `Cs`, and
`model_at(C)`, which returns a new, unfitted model at that `C`.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
folds = StratifiedKFold(5, shuffle=True, random_state=6103)
Cs = [0.001, 0.01, 0.1, 1, 10, 100, 10000]
def model_at(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=20000))

In [ ]:
train_acc, cv_acc = [], []
for C in Cs:
    pipe = model_at(C)
    train_acc.append(...)
    cv_acc.append(...)
best_C = ...
test_acc = ...

print(train_acc); print(cv_acc); print(best_C, test_acc)

In [ ]:
# self-check: run this after your answer above
from sklearn.model_selection import cross_val_score
wt, wc = [], []
for C_ in Cs:
    p_ = make_pipeline(StandardScaler(), LogisticRegression(C=C_, max_iter=20000))
    wt.append(p_.fit(X_train, y_train).score(X_train, y_train))
    wc.append(cross_val_score(p_, X_train, y_train, cv=folds).mean())
wb = Cs[int(np.argmax(wc))]
wtest = make_pipeline(StandardScaler(), LogisticRegression(C=wb, max_iter=20000)).fit(X_train, y_train).score(X_test, y_test)
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("train_acc", train_acc, lambda: np.allclose(train_acc, wt),
          "should be each model's accuracy on the data it was fit on, one entry per C."),
    _part("cv_acc", cv_acc, lambda: np.allclose(cv_acc, wc),
          "should be cross_val_score(pipe, X_train, y_train, cv=folds).mean(), one entry per C."),
    _part("best_C", best_C, lambda: best_C == wb, "should maximize the cross-validated accuracy, not the training accuracy."),
    _part("test_acc", test_acc, lambda: abs(float(test_acc) - wtest) < 1e-9,
          "should be the best_C model, refit on all of X_train and scored on X_test."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B4. What ridge shrinks, on real data

*Suggested: 12 minutes.*

`db` is the diabetes data: 442 patients, ten baseline measurements, and disease
progression a year later. Six of the measurements, `s1` to `s6`, are blood
serum readings, and some of them are nearly sums of others.

The features are standardized and centered for you. Using the SVD, as in
[the reading](https://www.smajhi.com/DATS-6103/regularization/regularization.html#what-ridge-shrinks-the-svd-view):

- `s`---the singular values of `Zc`;
- `f`---the shrink factors at `alpha = 10`;
- `w`---the ridge coefficients from the SVD formula.

Then compare `w` with `w_ols` on the features `s1` and `s2`.

**Already loaded:** `db`, the features `Zc`, the centered target `yc`, `alpha`, and
the least squares coefficients `w_ols`.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, LinearRegression
db = load_diabetes()
X, y = db.data, db.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103)
Z = StandardScaler().fit_transform(X_train)
Zc = Z - Z.mean(axis=0)
yc = y_train - y_train.mean()
alpha = 10.0
w_ols = LinearRegression().fit(Z, y_train).coef_

In [ ]:
U, s, Vt = ..., ..., ...
f = ...
w = ...

if s is not ...:
    print("singular values:", s.round(2))
if f is not ...:
    print("shrink factors: ", f.round(3))
if w is not ...:
    for name, a, b in zip(db.feature_names, w_ols, w):
        print(f"{name:>4}  OLS {a:7.2f}   ridge {b:7.2f}")

In [ ]:
# self-check: run this after your answer above
from sklearn.linear_model import Ridge
U_, s_, Vt_ = np.linalg.svd(Zc, full_matrices=False)
wf = s_**2 / (s_**2 + alpha)
wr = Ridge(alpha=alpha).fit(Z, y_train).coef_
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("s", s, lambda: np.allclose(np.asarray(s), s_), "should be the singular values from np.linalg.svd(Zc, full_matrices=False)."),
    _part("f", f, lambda: np.allclose(np.asarray(f), wf), "should be s**2 / (s**2 + alpha)."),
    _part("w", w, lambda: np.allclose(np.asarray(w), wr),
          "that is the OLS solution; ridge scales each direction by s / (s**2 + alpha)."
          if not _blank(w) and np.allclose(np.asarray(w), w_ols)
          else "should be Vt.T @ np.diag(s / (s**2 + alpha)) @ U.T @ yc."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B5. Bending the boundary

*Suggested: 10 minutes.*

[Polynomial logistic regression](https://www.smajhi.com/DATS-6103/regularization/regularization.html) on a real
flower. `ir` is the iris data: 150 flowers, three species. Using only petal length
and petal width, predict whether a flower is **versicolor** (`1`) or not (`0`).
Versicolor sits between the other two species, so try to picture a straight line
that separates it before you run anything.

- `baseline`---the test accuracy of always answering "not versicolor";
- `train_acc`, `cv_acc`---lists, one entry per degree in `degrees`, as in B3;
- `best_degree`---the degree with the highest cross-validated accuracy;
- `test_acc`---that model's accuracy on the test set.

**Already loaded:** `ir`, `X_train`, `X_test`, `y_train` and `y_test`, `folds`,
`degrees`, `n_weights` (how many weights each degree has, intercept included), and
`model_at(degree)`, which returns a new, unfitted model of that degree with a very
weak penalty ($C$ = 10,000).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
ir = load_iris()
X = ir.data[:, [2, 3]]
y = (ir.target == 1).astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103, stratify=y)
degrees = [1, 2, 3, 4, 6, 8]
n_weights = [(d + 1) * (d + 2) // 2 for d in degrees]
def model_at(degree):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(),
                         LogisticRegression(C=10000, max_iter=50000))

In [ ]:
baseline = ...

train_acc, cv_acc = [], []
for d in degrees:
    pipe = model_at(d)
    train_acc.append(...)
    cv_acc.append(...)

best_degree = ...
test_acc = ...

print("baseline", baseline)
for row in zip(degrees, n_weights, train_acc, cv_acc):
    print(*row)
print(best_degree, test_acc)

In [ ]:
# self-check: run this after your answer above
wt, wc = [], []
for d_ in degrees:
    p_ = make_pipeline(PolynomialFeatures(d_, include_bias=False), StandardScaler(),
                       LogisticRegression(C=10000, max_iter=50000))
    wt.append(p_.fit(X_train, y_train).score(X_train, y_train))
    wc.append(cross_val_score(p_, X_train, y_train, cv=folds).mean())
wb = degrees[int(np.argmax(wc))]
wtest = make_pipeline(PolynomialFeatures(wb, include_bias=False), StandardScaler(),
                      LogisticRegression(C=10000, max_iter=50000)).fit(X_train, y_train).score(X_test, y_test)
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("baseline", baseline, lambda: abs(float(baseline) - float((y_test == 0).mean())) < 1e-9,
          "should be the share of TEST flowers that are not versicolor."),
    _part("train_acc", train_acc, lambda: np.allclose(train_acc, wt),
          "should be each model's accuracy on the data it was fit on, one entry per degree."),
    _part("cv_acc", cv_acc, lambda: np.allclose(cv_acc, wc),
          "should be cross_val_score(pipe, X_train, y_train, cv=folds).mean(), one entry per degree."),
    _part("best_degree", best_degree, lambda: best_degree == wb,
          "should maximize the cross-validated accuracy, not the training accuracy."),
    _part("test_acc", test_acc, lambda: abs(float(test_acc) - wtest) < 1e-9,
          "should be the best_degree model, refit on all of X_train and scored on X_test."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

### B6. Ten classes at once

*Suggested: 10 minutes.*

Multinomial logistic regression, from
[the reading](https://www.smajhi.com/DATS-6103/logistic/multinomial.html), on handwritten digits: 1,797 images,
8 by 8 pixels, ten classes. `model` is written and fitted for you, with the scaler in
front, and `cm` is its confusion matrix. On the test data, report:

- `acc`---the accuracy;
- `recall`---an array of ten per-class recalls, digit 0 first;
- `worst`---the digit with the lowest recall;
- `confused`---the most common mistake, as a tuple `(true digit, predicted digit)`.

**Already loaded:** `dg`, `X_train`, `X_test`, `y_train` and `y_test`, and `confusion_matrix`.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
dg = load_digits()
X, y = dg.data, dg.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=6103, stratify=y)

In [ ]:
# The scaler is given. HW Regression Q4 shows why it matters.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_train, y_train)
cm = confusion_matrix(y_test, model.predict(X_test))   # rows: true, columns: predicted
acc = ...
recall = ...
worst = ...
off = cm.copy(); np.fill_diagonal(off, 0)
confused = ...

print(acc, recall, worst, confused)

In [ ]:
# self-check: run this after your answer above
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X_train, y_train)
c = confusion_matrix(y_test, m.predict(X_test))
wr = c.diagonal() / c.sum(axis=1)
o = c.copy(); np.fill_diagonal(o, 0)
def _blank(v):
    return (v is ... or (isinstance(v, (list, tuple)) and any(_blank(x) for x in v))
            or (isinstance(v, dict) and any(_blank(x) for x in v.values())))
def _part(name, value, test, hint):
    if _blank(value):
        return False, f"{name}: not filled in yet."
    try:
        good = bool(test())
    except Exception:
        good = False
    return good, (f"{name}: right." if good else f"{name}: {hint}")
globals().update(locals())                 # the page runs checks with separate globals and locals
_parts = [
    _part("acc", acc, lambda: abs(float(acc) - c.trace() / c.sum()) < 1e-9,
          "should be the share of the diagonal: cm.trace() / cm.sum()."),
    _part("recall", recall, lambda: np.allclose(np.asarray(recall), wr),
          "should be the diagonal divided by each ROW's total: rows are the true digits."),
    _part("worst", worst, lambda: int(worst) == int(wr.argmin()), "should be the digit with the lowest recall."),
    _part("confused", confused,
          lambda: tuple(int(v) for v in confused) == tuple(int(v) for v in np.unravel_index(o.argmax(), o.shape)),
          "should be (row, column) of the largest off-diagonal count: np.unravel_index(off.argmax(), off.shape)."),
]
assert all(g for g, _ in _parts), " ".join(m for _, m in _parts)
print("looks right")

## Before you leave

Upload this notebook to Blackboard. It is the record of what you did; the lab
grade itself comes from what you do in the room.

Then post a comment at the bottom of the lab page, <https://www.smajhi.com/DATS-6103/labs/regression.html>. One line is enough.
It proves your GitHub account works with the comment system, which is how every
week's reading credit is earned.